In [4607]:
# تحميل المكتبات الأساسية
import pandas as pd
import numpy as np

# مكتبة التعامل مع البيانات المفقودة وتوحيد القيم
from sklearn.impute import SimpleImputer
# مكتبة لتنظيف البيانات وتعديل الأعمدة
import re

In [4608]:
sales_transactions = pd.read_csv('Sales_Transactions.csv')
store_information = pd.read_csv('Store_Information.csv')
customer_data = pd.read_csv('Customer_Data.csv')
marketing_campaigns = pd.read_csv('Marketing_Campaigns.csv')
store_complaints = pd.read_csv('Store_Complaints.csv')


In [4609]:
# check the data shape
print(sales_transactions.shape)

(25375, 8)


In [4610]:
# استعراض أول 5 صفوف
sales_transactions.head()
#اكتشفت فيه قيم سالبه في الايرادات اما مرتجعات او خطا في الادخال  
# لكن لن اعتبرها مرتجعات لان كل قيم الوحدات المباعه بالموجب ايضا لاتعتبر دفع اجل او دين
#  لانه بعض الايرادات السالبه ليس لها كستمر اي دي

,Transaction_ID,Store_ID,Date,Product_Category,Units_Sold,Revenue,Discount_Percentage,Customer_ID
0,TXN023368,STORE_027,20/05/2024,Electronics,9,"-1,794",15,CUST00836
1,TXN004823,STORE_039,19/04/2023,Home & Garden,11,373,5,CUST06683
2,TXN024939,STORE_029,2023-03-04,Food & Beverage,4,"4,268",15,CUST03565
3,TXN015257,STORE_033,06/03/2024,Electronics,39,"4,325",0,CUST07318
4,TXN024609,STORE_021,2023-05-30,Electronics,42,"4,409",0,CUST07210


In [4611]:
# عرض معلومات عن الأعمدة
sales_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25375 entries, 0 to 25374
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Transaction_ID       25375 non-null  object 
 1   Store_ID             25375 non-null  object 
 2   Date                 25375 non-null  object 
 3   Product_Category     25375 non-null  object 
 4   Units_Sold           25375 non-null  int64  
 5   Revenue              25375 non-null  float64
 6   Discount_Percentage  25375 non-null  int64  
 7   Customer_ID          23344 non-null  object 
dtypes: float64(1), int64(2), object(5)
memory usage: 1.5+ MB


In [4612]:
# تحليل القيم المفقودة في الأعمدة
sales_transactions.isnull().sum()

Transaction_ID            0
Store_ID                  0
Date                      0
Product_Category          0
Units_Sold                0
Revenue                   0
Discount_Percentage       0
Customer_ID            2031
dtype: int64

In [4613]:
pd.options.display.float_format = '{:,.0f}'.format

# إحصائيات وصفية عن البيانات الرقمية
sales_transactions.describe()

,Units_Sold,Revenue,Discount_Percentage
count,"25,375","25,375","25,375"
mean,25,"2,537",9
std,15,"2,885",9
min,1,"-4,989",0
25%,13,"1,225",0
50%,25,"2,504",10
75%,37,"3,742",15
max,240,"98,062",30


In [4614]:
# التحقق من وجود مسافات فارغة فقط في الأعمدة النصية
whitespace_rows = sales_transactions[sales_transactions.select_dtypes(include=['object']).apply(lambda x: x.str.contains('^\s*$', na=False)).any(axis=1)]

# عرض الصفوف التي تحتوي على مسافات فارغة في الأعمدة النصية
print("\nRows with White Spaces in Any Text Column:")
print(whitespace_rows)


Rows with White Spaces in Any Text Column:
Empty DataFrame
Columns: [Transaction_ID, Store_ID, Date, Product_Category, Units_Sold, Revenue, Discount_Percentage, Customer_ID]
Index: []


In [4615]:
# عرض القيم الفريدة في عمود Product_Category
sales_transactions['Product_Category'].unique()

array(['Electronics', 'Home & Garden', 'Food & Beverage', 'Beauty',
       'Clothing', 'Sports'], dtype=object)

In [4616]:
# اتضح عندنا قيم مفقوده في customer ID 
# وايضا قيم بالسالب في Revenue 
#لن نحذف صفوف الي فيها قيم مفقوده للكستمر اي دي لاننا كذا بنخسر 8% من بيانات المبيعات بدون سبب ولايهمني الان تحليل يخص العميل 
#صيغ التاريخ ليست موحده
#هنالك 375 صف مكرر في 	Transaction_ID ببعض الفروق في الايردات فروق في الكسور ايضا في الكستمر اي دي بعضها فيه nan



In [4617]:
# تحويل العمود Date إلى نوع datetime مع توحيد الصيغ في DataFrame الصحيح
sales_transactions['Date'] = pd.to_datetime(sales_transactions['Date'], errors='coerce')



C:\Users\POWER\AppData\Local\Temp\ipykernel_35936\3307575795.py:2: UserWarning: Parsing dates in DD/MM/YYYY format when dayfirst=False (the default) was specified. This may lead to inconsistently parsed dates! Specify a format to ensure consistent parsing.
  sales_transactions['Date'] = pd.to_datetime(sales_transactions['Date'], errors='coerce')


In [ ]:

# التحقق من وجود تواريخ قبل 2023 أو بعد يونيو 2024
invalid_dates = sales_transactions[(sales_transactions['Date'] < '2023-01-01') | (sales_transactions['Date'] > '2024-06-30')]

# عرض التواريخ غير المتوافقة
print(f"Number of rows with invalid dates: {invalid_dates.shape[0]}")
print(invalid_dates[['Transaction_ID', 'Date']].head())

# عرض أقدم وأحدث تاريخ
earliest_date = sales_transactions['Date'].min()
latest_date = sales_transactions['Date'].max()

print(f"\nEarliest Date in the data: {earliest_date}")
print(f"Latest Date in the data: {latest_date}")
#بملف التعليمات مكتوب ان البيانات من 
#2023 يناير الى 2024 يونيو
# هنا تظهر 493 عمليه خارج هذه المده 
#قررت ابقي على هذه البيانات لانها تدعم القرار

Number of rows with invalid dates: 493
    Transaction_ID       Date
14       TXN009718 2024-08-05
120      TXN022087 2024-08-06
152      TXN016199 2024-07-04
195      TXN003762 2024-07-02
213      TXN014862 2024-11-01

Earliest Date in the data: 2023-01-01 00:00:00
Latest Date in the data: 2024-12-06 00:00:00


In [4619]:
# تحويل جميع القيم السلبية في Revenue إلى قيم موجبة
sales_transactions['Revenue'] = sales_transactions['Revenue'].abs()

# عرض أول 5 صفوف بعد التعديل للتأكد
print("\nSales Transactions (After Making Revenue Positive):")
sales_transactions.head()
# #حولناها لموجب ولم نعتبرها مرتجعات لان ماعندي المعلومه الكافيه اذا كان مرتجع ام لا 
# وكذلك هدفي معرفة اين افضل 5 مناطق للفتح فيها والمرتجعات لاتهمنا الان 
#يهمنا الان كم اجمالي الايرادات وليست الارباح 


Sales Transactions (After Making Revenue Positive):


,Transaction_ID,Store_ID,Date,Product_Category,Units_Sold,Revenue,Discount_Percentage,Customer_ID
0,TXN023368,STORE_027,2024-05-20,Electronics,9,"1,794",15,CUST00836
1,TXN004823,STORE_039,2023-04-19,Home & Garden,11,373,5,CUST06683
2,TXN024939,STORE_029,2023-03-04,Food & Beverage,4,"4,268",15,CUST03565
3,TXN015257,STORE_033,2024-06-03,Electronics,39,"4,325",0,CUST07318
4,TXN024609,STORE_021,2023-05-30,Electronics,42,"4,409",0,CUST07210


In [4620]:
# التحقق من التكرار في Transaction_ID
duplicate_transactions = sales_transactions[sales_transactions.duplicated(subset='Transaction_ID', keep=False)]


duplicate_transactions.head(10)

# تحديد الصفوف المكررة بناءً على Transaction_ID



,Transaction_ID,Store_ID,Date,Product_Category,Units_Sold,Revenue,Discount_Percentage,Customer_ID
4,TXN024609,STORE_021,2023-05-30,Electronics,42,"4,409",0,CUST07210
19,TXN004269,STORE_018,2024-02-04,Clothing,26,123,15,CUST00906
34,TXN019979,STORE_045,2023-05-06,Electronics,19,"1,756",10,CUST00691
100,TXN002433,STORE_044,2024-05-26,Sports,39,"2,994",20,CUST06467
143,TXN023878,STORE_042,2024-01-08,Home & Garden,48,"2,073",10,CUST04039
148,TXN023358,STORE_031,2023-03-21,Beauty,22,"2,949",0,CUST07246
158,TXN018764,STORE_041,2023-08-10,Beauty,2,655,25,CUST00191
171,TXN011504,STORE_008,2023-10-07,Clothing,14,"4,567",0,CUST02684
244,TXN004458,STORE_042,2023-07-20,Beauty,38,"3,923",10,CUST06007
393,TXN021358,STORE_016,2023-01-09,Home & Garden,28,"1,147",5,CUST01691


In [4621]:
# حساب عدد القيم الفريدة في Transaction_ID
unique_transaction_ids = sales_transactions['Transaction_ID'].nunique()

# عرض عدد القيم الفريدة
print(f"عدد القيم الفريدة في Transaction_ID: {unique_transaction_ids}")

عدد القيم الفريدة في Transaction_ID: 25000


In [4622]:
#نشوف اذا متكرر الادخال او هنالك اختلاف في الاعمده الثانيه لل375 صف متكرر من Transaction_ID  
# التحقق من الصفوف المكررة في جميع الأعمدة
duplicate_rows = sales_transactions[sales_transactions.duplicated(keep=False)]

# عرض الصفوف المتكررة
print("Duplicate Rows (Exact Matches in All Columns):")
print(duplicate_rows)


Duplicate Rows (Exact Matches in All Columns):
Empty DataFrame
Columns: [Transaction_ID, Store_ID, Date, Product_Category, Units_Sold, Revenue, Discount_Percentage, Customer_ID]
Index: []


In [4623]:
#  حساب عدد تكرار كل Transaction_ID
transaction_counts = sales_transactions['Transaction_ID'].value_counts()

# عرض عدد التكرار لكل Transaction_ID
print("Transaction_ID Counts (How many times each ID appears):\n", transaction_counts)
print("\n max repetaions",transaction_counts.max())
#نتاكد انه لم يتكرر اكثر من مره فطلع لنا التكرار مره واحده 

Transaction_ID Counts (How many times each ID appears):
 TXN003495    2
TXN006479    2
TXN015451    2
TXN016393    2
TXN001910    2
TXN003733    2
TXN016975    2
TXN008501    2
TXN014048    2
TXN002541    2
TXN007204    2
TXN023708    2
TXN006962    2
TXN005918    2
TXN003927    2
TXN012516    2
TXN016651    2
TXN003816    2
TXN008771    2
TXN013452    2
TXN022876    2
TXN002549    2
TXN000606    2
TXN010001    2
TXN008409    2
TXN023026    2
TXN003741    2
TXN018432    2
TXN021109    2
TXN003971    2
TXN013356    2
TXN001375    2
TXN011565    2
TXN024808    2
TXN007263    2
TXN001527    2
TXN018297    2
TXN014242    2
TXN016842    2
TXN006018    2
TXN017009    2
TXN001186    2
TXN007231    2
TXN018345    2
TXN024066    2
TXN009480    2
TXN011837    2
TXN009691    2
TXN017078    2
TXN023318    2
TXN009622    2
TXN023271    2
TXN019470    2
TXN015908    2
TXN023832    2
TXN010718    2
TXN013014    2
TXN019195    2
TXN010480    2
TXN015152    2
TXN003207    2
TXN009387    2
TXN013233    

In [4624]:
# تلخيص الاختلافات بناءً على جميع الأعمدة
differences = []  # يجب أن تحتوي على البيانات المكررة مع الاختلافات في الأعمدة

# تلخيص الفروقات في الأعمدة
column_diff_count = {col: 0 for col in sales_transactions.columns}  # العد لكل عمود

for transaction_id, group in sales_transactions.groupby('Transaction_ID'):
    # إذا كانت هناك أكثر من صف مع نفس Transaction_ID
    if len(group) > 1:
        diff_columns = {}
        for column in group.columns:
            if group[column].nunique() > 1:  # إذا كانت هناك اختلافات في العمود
                diff_columns[column] = group[column].unique()
                column_diff_count[column] += 1
        differences.append((transaction_id, diff_columns))

# تلخيص النتائج
print("\nSummary of Differences:")
for column, count in column_diff_count.items():
    print(f"Cases with different {column}: {count}")


Summary of Differences:
Cases with different Transaction_ID: 0
Cases with different Store_ID: 0
Cases with different Date: 0
Cases with different Product_Category: 0
Cases with different Units_Sold: 0
Cases with different Revenue: 375
Cases with different Discount_Percentage: 0
Cases with different Customer_ID: 0


In [4625]:
# تحديد الصفوف المكررة بناءً على Transaction_ID
duplicate_transactions = sales_transactions[sales_transactions.duplicated(subset='Transaction_ID', keep=False)]

# عرض قيم Revenue لكل Transaction_ID مكرر
revenue_values = duplicate_transactions.groupby('Transaction_ID')['Revenue'].unique()

# عرض قيم Revenue لكل Transaction_ID
print("Revenue Values for Duplicate Transaction_IDs:\n", revenue_values)

#اتضح انه اختلاف فقط في الايرادات والاختلاف عباره عن كسور وبسيط جدا فلذلك بناخذ العدد المقرب وليس اللي فيه كسور كثيره 
# حذف 375 صف من الملف 
#قررت اخذ اللي عنده قيمة الايرادات النظيف بدون كسور واللي له كسور كثيره يتم حذفه لان الرقم النظيف هو اقرب للفاتوره الصحيحه 


Revenue Values for Duplicate Transaction_IDs:
 Transaction_ID
TXN000055    [3101.6627528600447, 3103.35]
TXN000098    [3322.71, 3332.4818213267663]
TXN000203     [171.32733298231742, 180.11]
TXN000204    [2277.9736530932573, 2280.57]
TXN000364      [319.5450492693837, 324.96]
TXN000416      [2393.1, 2402.756501891008]
TXN000444     [2205.22, 2209.356229607718]
TXN000574    [3339.67, 3337.7897662801506]
TXN000606    [2421.89, 2423.3357684359703]
TXN000653      [398.68, 398.8544601807936]
TXN000746      [4535.059110704508, 4527.1]
TXN000786    [4012.03, 4003.4434535858154]
TXN000834      [1230.04, 1233.16005215549]
TXN000859      [560.84, 568.3523054060678]
TXN000870    [2367.11, 2374.1308214873134]
TXN000874      [762.14, 762.4185634064594]
TXN000987     [2545.54, 2548.915424897569]
TXN001109      [734.4361586764275, 741.21]
TXN001115     [2736.552801148621, 2727.02]
TXN001117     [1626.56, 1618.581538814822]
TXN001173      [743.13, 742.8728949746045]
TXN001174    [2034.02, 2026.1932614

In [4626]:


# تحديد الصفوف المكررة بناءً على Transaction_ID
duplicate_transactions = sales_transactions[sales_transactions.duplicated(subset='Transaction_ID', keep=False)]

# تصفية الصفوف المكررة التي تحتوي على إيرادات بأرقام عشرية كثيرة (أكثر من منزلتين عشريتين)
rows_with_large_decimals = duplicate_transactions[duplicate_transactions['Revenue'].apply(lambda x: len(str(x).split('.')[-1]) > 2)]

# حذف الصفوف المكررة التي تحتوي على إيرادات بكسور كثيرة
sales_transactions= sales_transactions.drop(rows_with_large_decimals.index)



# عرض عدد الصفوف بعد الحذف للتأكد
print(sales_transactions.shape)

(25000, 8)


In [4627]:
sales_transactions.head()

,Transaction_ID,Store_ID,Date,Product_Category,Units_Sold,Revenue,Discount_Percentage,Customer_ID
0,TXN023368,STORE_027,2024-05-20,Electronics,9,"1,794",15,CUST00836
1,TXN004823,STORE_039,2023-04-19,Home & Garden,11,373,5,CUST06683
2,TXN024939,STORE_029,2023-03-04,Food & Beverage,4,"4,268",15,CUST03565
3,TXN015257,STORE_033,2024-06-03,Electronics,39,"4,325",0,CUST07318
4,TXN024609,STORE_021,2023-05-30,Electronics,42,"4,409",0,CUST07210


In [4628]:
# التحقق من وجود الصفوف المكررة بناءً على Transaction_ID بعد الحذف
duplicate_transaction_ids = sales_transactions[sales_transactions.duplicated(subset='Transaction_ID', keep=False)]

# عرض التكرارات المتبقية (إذا كانت موجودة)
print(f"عدد الصفوف المكررة المتبقية: {duplicate_transaction_ids.shape[0]}")

# عرض بعض الصفوف المكررة المتبقية
print("\nRows with Duplicate Transaction_IDs after cleaning:")
print(duplicate_transaction_ids.head())

عدد الصفوف المكررة المتبقية: 0

Rows with Duplicate Transaction_IDs after cleaning:
Empty DataFrame
Columns: [Transaction_ID, Store_ID, Date, Product_Category, Units_Sold, Revenue, Discount_Percentage, Customer_ID]
Index: []


In [4629]:
## نبدا في الملف الثاني


In [4630]:
# check the data shape
print(store_information.shape)

(46, 7)


In [4631]:
# استعراض صفوف

store_information.head(46)

,Store_ID,Region,City,Store_Size_SQM,Opening_Date,Staff_Count,Monthly_Rent
0,Store_001,Central,Riyadh,1107,30/7/2021,23,"115,878"
1,Store_002,Eastern,Jubail,350,27/3/2023,44,"34,482"
2,Store_003,Northern,Arar,609,30/7/2021,49,"39,420"
3,Store_004,Central,Qassim,496,6/7/2020,11,"56,494"
4,Store_005,Central,Qassim,1543,17/2/2023,9,"133,174"
5,Store_006,Western,Taif,1558,18/2/2022,24,NaN
6,Store_007,Western,Jeddah,1212,27/4/2020,17,"87,844"
7,Store_008,Western,Medina,554,4/8/2022,12,"131,482"
8,Store_009,Western,Medina,492,1/9/2021,36,"27,439"
9,Store_010,Northern,Arar,1562,30/9/2023,41,"37,524"


In [4632]:

# عرض معلومات عن الأعمدة

store_information.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46 entries, 0 to 45
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Store_ID        46 non-null     object 
 1   Region          46 non-null     object 
 2   City            46 non-null     object 
 3   Store_Size_SQM  46 non-null     int64  
 4   Opening_Date    46 non-null     object 
 5   Staff_Count     46 non-null     int64  
 6   Monthly_Rent    42 non-null     float64
dtypes: float64(1), int64(2), object(4)
memory usage: 2.6+ KB


In [4633]:
# فحص القيم المفقودة في الأعمدة
print("\nMissing values in Store Information:")
store_information.isnull().sum()


Missing values in Store Information:


Store_ID          0
Region            0
City              0
Store_Size_SQM    0
Opening_Date      0
Staff_Count       0
Monthly_Rent      4
dtype: int64

In [4634]:


# عرض الإحصائيات الوصفية للبيانات الرقمية
print("\nDescriptive Statistics (Store Information):")
store_information.describe()


Descriptive Statistics (Store Information):


,Store_Size_SQM,Staff_Count,Monthly_Rent
count,46,46,42
mean,"1,136",25,"100,342"
std,519,13,"42,863"
min,274,5,"21,729"
25%,630,14,"61,602"
50%,"1,160",24,"107,584"
75%,"1,561",34,"132,751"
max,"1,941",50,"170,283"


In [4635]:
# حساب القيم الفريدة في City و Region
unique_cities = store_information['City'].unique()
unique_regions = store_information['Region'].unique()

# حساب عدد القيم الفريدة في Store_ID
unique_store_ids_count = store_information['Store_ID'].nunique()

# عرض النتائج
print(f"   City : {unique_cities}")


print(f"  Regions : {unique_regions}")


print(f"store_id : {unique_store_ids_count}")


   City : ['Riyadh' 'Jubail' 'Arar' 'Qassim' 'Taif' 'Jeddah' 'Medina' 'Makkah'
 'Khobar' 'Dhahran' 'Tabuk' 'Abha' 'Dammam' 'Jizan' 'Hail' 'Najran'
 'Duplicate_Error_City']
  Regions : ['Central' 'Eastern' 'Northern' 'Western' 'Southern']
store_id : 45


In [4636]:
# تعريف قائمة المدن والمناطق الصحيحة
city_region_map = {
    'Riyadh': 'Central', 
    'Jubail': 'Eastern', 
    'Arar': 'Northern', 
    'Qassim': 'Central', 
    'Taif': 'Western', 
    'Jeddah': 'Western', 
    'Medina': 'Western', 
    'Makkah': 'Western',
    'Khobar': 'Eastern', 
    'Dhahran': 'Eastern', 
    'Tabuk': 'Northern', 
    'Abha': 'Southern', 
    'Dammam': 'Eastern', 
    'Jizan': 'Southern', 
    'Hail': 'Central', 
    'Najran': 'Southern',
    
}

# التحقق من أن كل مدينة تتبع المنطقة الصحيحة
store_info_invalid_regions = store_information[store_information.apply(
    lambda row: city_region_map.get(row['City'], 'Unknown') != row['Region'], axis=1)]

# عرض الصفوف التي تحتوي على تباس بين المدينة والمنطقة
print(f"Number of rows with mismatched city-region: {store_info_invalid_regions.shape[0]}")
print(store_info_invalid_regions[['Store_ID', 'City', 'Region']].head(10))

Number of rows with mismatched city-region: 1
     Store_ID                  City   Region
45  Store_006  Duplicate_Error_City  Western


In [4637]:
# التحقق من وجود مسافات فارغة فقط في الأعمدة النصية
whitespace_rows = store_information[store_information.select_dtypes(include=['object']).apply(lambda x: x.str.contains('^\s*$', na=False)).any(axis=1)]

# عرض الصفوف التي تحتوي على مسافات فارغة في الأعمدة النصية
print("\nRows with White Spaces in Any Text Column:")
print(whitespace_rows)


Rows with White Spaces in Any Text Column:
Empty DataFrame
Columns: [Store_ID, Region, City, Store_Size_SQM, Opening_Date, Staff_Count, Monthly_Rent]
Index: []


In [4638]:
#عندي 4 قيم مفقوده في الايجار الشهري 
#كذلك تحويل نوع البيانات الى date  
#توحيد صيغة البيانات في عمود ال store id 

In [4639]:
# تعديل دالة توحيد Store_ID
def standardize_store_id(store_id):
    # استخراج الرقم فقط من Store_ID
    store_id = re.sub(r'\D', '', str(store_id))  # إزالة أي حروف أو رموز غير رقمية
    return f"STORE_{store_id.zfill(3)}"  # توحيد التنسيق ليكون STORE_XXX

# تطبيق التوحيد على العمود Store_ID
store_information['Store_ID'] = store_information['Store_ID'].apply(standardize_store_id)

# التحقق من عدم وجود تكرار في Store_ID
duplicate_store_ids = store_information[store_information.duplicated(subset='Store_ID')]

# عرض Store_ID المكررة
print("Duplicate Store_IDs:\n", duplicate_store_ids)





Duplicate Store_IDs:
      Store_ID   Region                  City  Store_Size_SQM Opening_Date  \
45  STORE_006  Western  Duplicate_Error_City            1558    18/2/2022   

    Staff_Count  Monthly_Rent  
45           24        98,319  


In [4640]:
# عرض أول 5 صفوف بعد توحيد Store_ID
print("\nStore Information (After Standardizing Store_ID):")
store_information.head(5)


Store Information (After Standardizing Store_ID):


,Store_ID,Region,City,Store_Size_SQM,Opening_Date,Staff_Count,Monthly_Rent
0,STORE_001,Central,Riyadh,1107,30/7/2021,23,"115,878"
1,STORE_002,Eastern,Jubail,350,27/3/2023,44,"34,482"
2,STORE_003,Northern,Arar,609,30/7/2021,49,"39,420"
3,STORE_004,Central,Qassim,496,6/7/2020,11,"56,494"
4,STORE_005,Central,Qassim,1543,17/2/2023,9,"133,174"


In [4641]:
#وجدنا قيمة الايجار  المفقوده في احد الصفوف المتكرره 
# تحديد الصفوف المكررة بناءً على Store_ID
duplicates = store_information[store_information.duplicated(subset='Store_ID', keep='first')]

# عرض الصفوف المكررة لتحديد المشكلة
print("Duplicate Store_IDs:")
print(duplicates)

# التعامل مع الصفوف المكررة:
# سنملأ NaN في Monthly_Rent من الصف الآخر الذي يحتوي على القيمة الصحيحة
store_information['Monthly_Rent'] = store_information.groupby('Store_ID')['Monthly_Rent'].transform(lambda x: x.fillna(method='bfill').fillna(method='ffill'))

# حذف الصف المكرر (الذي أخذنا منه القيمة)
store_information = store_information.drop(duplicates.index)



Duplicate Store_IDs:
     Store_ID   Region                  City  Store_Size_SQM Opening_Date  \
45  STORE_006  Western  Duplicate_Error_City            1558    18/2/2022   

    Staff_Count  Monthly_Rent  
45           24        98,319  


In [4642]:
# تحويل العمود Date إلى نوع datetime مع توحيد الصيغ في DataFrame الصحيح
store_information['Opening_Date'] = pd.to_datetime(store_information['Opening_Date'], errors='coerce')

C:\Users\POWER\AppData\Local\Temp\ipykernel_35936\358555093.py:2: UserWarning: Parsing dates in DD/MM/YYYY format when dayfirst=False (the default) was specified. This may lead to inconsistently parsed dates! Specify a format to ensure consistent parsing.
  store_information['Opening_Date'] = pd.to_datetime(store_information['Opening_Date'], errors='coerce')


In [4643]:

# عرض أقدم وأحدث تاريخ
earliest_date = store_information['Opening_Date'].min()
latest_date = store_information['Opening_Date'].max()

print(f"\nEarliest Date in the data: {earliest_date}")
print(f"Latest Date in the data: {latest_date}")


Earliest Date in the data: 2020-01-11 00:00:00
Latest Date in the data: 2023-11-24 00:00:00


In [4644]:
store_information.head(46)

,Store_ID,Region,City,Store_Size_SQM,Opening_Date,Staff_Count,Monthly_Rent
0,STORE_001,Central,Riyadh,1107,2021-07-30,23,"115,878"
1,STORE_002,Eastern,Jubail,350,2023-03-27,44,"34,482"
2,STORE_003,Northern,Arar,609,2021-07-30,49,"39,420"
3,STORE_004,Central,Qassim,496,2020-06-07,11,"56,494"
4,STORE_005,Central,Qassim,1543,2023-02-17,9,"133,174"
5,STORE_006,Western,Taif,1558,2022-02-18,24,"98,319"
6,STORE_007,Western,Jeddah,1212,2020-04-27,17,"87,844"
7,STORE_008,Western,Medina,554,2022-04-08,12,"131,482"
8,STORE_009,Western,Medina,492,2021-01-09,36,"27,439"
9,STORE_010,Northern,Arar,1562,2023-09-30,41,"37,524"


In [4645]:
# حساب متوسط الإيجار لكل مدينة
city_mean_rent = store_information.groupby('City')['Monthly_Rent'].mean()

# ملء القيم المفقودة في Monthly_Rent باستخدام المتوسط الخاص بكل مدينة
def fill_rent(row):
    if pd.isna(row['Monthly_Rent']):
        return city_mean_rent[row['City']]
    else:
        return row['Monthly_Rent']

# تطبيق التعديل على العمود Monthly_Rent
store_information['Monthly_Rent'] = store_information.apply(fill_rent, axis=1).round(2)



In [4646]:
store_information.head(45)

,Store_ID,Region,City,Store_Size_SQM,Opening_Date,Staff_Count,Monthly_Rent
0,STORE_001,Central,Riyadh,1107,2021-07-30,23,"115,878"
1,STORE_002,Eastern,Jubail,350,2023-03-27,44,"34,482"
2,STORE_003,Northern,Arar,609,2021-07-30,49,"39,420"
3,STORE_004,Central,Qassim,496,2020-06-07,11,"56,494"
4,STORE_005,Central,Qassim,1543,2023-02-17,9,"133,174"
5,STORE_006,Western,Taif,1558,2022-02-18,24,"98,319"
6,STORE_007,Western,Jeddah,1212,2020-04-27,17,"87,844"
7,STORE_008,Western,Medina,554,2022-04-08,12,"131,482"
8,STORE_009,Western,Medina,492,2021-01-09,36,"27,439"
9,STORE_010,Northern,Arar,1562,2023-09-30,41,"37,524"


In [4647]:
# حفظ النسخة المعدلة من Sales_Transactions
sales_transactions.to_csv('Cleaned_Sales_Transactions.csv', index=False)

# حفظ النسخة المعدلة من Store_Information
store_information.to_csv('Cleaned_Store_Information.csv', index=False)



In [4648]:
Cleaned_Store_Information = pd.read_csv('Cleaned_Store_Information.csv')
Cleaned_Sales_Transactions = pd.read_csv('Cleaned_Sales_Transactions.csv')


In [4649]:
Cleaned_Store_Information.shape


(45, 7)

In [4650]:
Cleaned_Sales_Transactions.shape

(25000, 8)

In [4651]:
# نبدا بالملف الثالث

In [4652]:
customer_data.shape

(8000, 7)

In [4653]:
customer_data.head()

,Customer_ID,Age,Gender,City,Registration_Date,Loyalty_Member,Total_Lifetime_Spend
0,CUST00001,31,M,Khobar,2022-10-24,Yes,"6,369"
1,CUST00002,63,F,Hail,2022-06-15,No,"48,421"
2,CUST00003,58,F,Jizan,2023-11-18,Yes,"32,933"
3,CUST00004,64,F,Jeddah,2023-03-01,Yes,"20,377"
4,CUST00005,47,F,Abha,2023-08-11,No,"46,611"


In [4654]:
customer_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Customer_ID           8000 non-null   object 
 1   Age                   7600 non-null   float64
 2   Gender                7600 non-null   object 
 3   City                  8000 non-null   object 
 4   Registration_Date     8000 non-null   object 
 5   Loyalty_Member        8000 non-null   object 
 6   Total_Lifetime_Spend  8000 non-null   float64
dtypes: float64(2), object(5)
memory usage: 437.6+ KB


In [4655]:
customer_data.isnull().sum()

Customer_ID               0
Age                     400
Gender                  400
City                      0
Registration_Date         0
Loyalty_Member            0
Total_Lifetime_Spend      0
dtype: int64

In [4656]:
customer_data.describe()

,Age,Total_Lifetime_Spend
count,"7,600","8,000"
mean,45,"24,966"
std,16,"14,424"
min,18,102
25%,31,"12,659"
50%,45,"25,083"
75%,60,"37,455"
max,74,"49,999"


In [4657]:
# التحقق من وجود مسافات فارغة فقط في الأعمدة النصية
whitespace_rows = customer_data[customer_data.select_dtypes(include=['object']).apply(lambda x: x.str.contains('^\s*$', na=False)).any(axis=1)]

# عرض الصفوف التي تحتوي على مسافات فارغة في الأعمدة النصية
print("\nRows with White Spaces in Any Text Column:")
print(whitespace_rows)


Rows with White Spaces in Any Text Column:
Empty DataFrame
Columns: [Customer_ID, Age, Gender, City, Registration_Date, Loyalty_Member, Total_Lifetime_Spend]
Index: []


In [4658]:
# حساب القيم الفريدة في City و gender
unique_cities = customer_data['City'].unique()
unique_Gender = customer_data['Gender'].unique()
unique_Loyalty_Member=customer_data['Loyalty_Member'].unique()
unique_Customer_ID_count = customer_data['Customer_ID'].nunique()

# عرض النتائج
print(f"   City : {unique_cities}")


print(f"  Gender : {unique_Gender}")
print(f" Loyalty_Member :{unique_Loyalty_Member}")
print(f"Customer_ID : {unique_Customer_ID_count}")

   City : ['Khobar' 'Hail' 'Jizan' 'Jeddah' 'Abha' 'Arar' 'Jubail' 'Taif' 'Qassim'
 'Najran' 'Riyadh' 'Mecca' 'Dammam' 'Tabuk' 'Dhahran' 'Medina']
  Gender : ['M' 'F' nan]
 Loyalty_Member :['Yes' 'No']
Customer_ID : 8000


In [4659]:
# تحويل العمود Date إلى نوع datetime مع توحيد الصيغ في DataFrame الصحيح
customer_data['Registration_Date'] = pd.to_datetime(customer_data['Registration_Date'], errors='coerce')


In [4660]:

# عرض أقدم وأحدث تاريخ
earliest_date = customer_data['Registration_Date'].min()
latest_date = customer_data['Registration_Date'].max()

print(f"\nEarliest Date in the data: {earliest_date}")
print(f"Latest Date in the data: {latest_date}")


Earliest Date in the data: 2022-01-01 00:00:00
Latest Date in the data: 2024-07-15 00:00:00


In [4661]:
#الان عندي 400 قيمه مفقود في الgender و 400 قيمه مفقوده في الage
#العمر المفقود تم استبداله بالمتوسط لكل مدينه والجنس تم استبداله بغير معروف
# Replace missing values in Gender with "Unknown"
customer_data['Gender'].fillna('Unknown', inplace=True)

# Display the number of missing values after replacement
print(f"Number of missing values in Gender after replacement: {customer_data['Gender'].isnull().sum()}")
# Calculate the average age for each city
age_by_city = customer_data.groupby('City')['Age'].mean().round()


print(age_by_city)
# Replace missing values in Age with the average age of each city
customer_data['Age'] = customer_data.apply(
    lambda row: age_by_city[row['City']] if pd.isnull(row['Age']) else row['Age'], axis=1)

# Display the number of missing values in Age after replacement
print(f"Number of missing values in Age after replacement: {customer_data['Age'].isnull().sum()}")

Number of missing values in Gender after replacement: 0
City
Abha      45
Arar      46
Dammam    47
Dhahran   45
Hail      45
Jeddah    45
Jizan     45
Jubail    45
Khobar    46
Mecca     46
Medina    45
Najran    45
Qassim    45
Riyadh    47
Tabuk     45
Taif      46
Name: Age, dtype: float64
Number of missing values in Age after replacement: 0


In [4662]:
# تحويل العمود Age من float إلى int
customer_data['Age'] = customer_data['Age'].astype('Int64')  

# التحقق من التعديل
print(f"Data types after conversion:\n{customer_data.dtypes}")

Data types after conversion:
Customer_ID                     object
Age                              Int64
Gender                          object
City                            object
Registration_Date       datetime64[ns]
Loyalty_Member                  object
Total_Lifetime_Spend           float64
dtype: object


In [4663]:
customer_data.to_csv('Cleaned_customer_data.csv', index=False)


In [4664]:
customer_data.head()

,Customer_ID,Age,Gender,City,Registration_Date,Loyalty_Member,Total_Lifetime_Spend
0,CUST00001,31,M,Khobar,2022-10-24,Yes,"6,369"
1,CUST00002,63,F,Hail,2022-06-15,No,"48,421"
2,CUST00003,58,F,Jizan,2023-11-18,Yes,"32,933"
3,CUST00004,64,F,Jeddah,2023-03-01,Yes,"20,377"
4,CUST00005,47,F,Abha,2023-08-11,No,"46,611"


In [4665]:
#الملف الرابع

In [4666]:
marketing_campaigns.shape

(218, 6)

In [4667]:
marketing_campaigns.head()

,Campaign_ID,Store_ID,Month,Campaign_Type,Budget_SAR,Impressions
0,CMP0001,STORE_033,2023-01,Influencer,46000 SAR,6758300
1,CMP0002,STORE_002,2023-01,Influencer,26000,47762300
2,CMP0003,STORE_002,2023-01,Outdoor,"40,000",30217100
3,CMP0004,STORE_045,2023-01,Outdoor,47000,38244400
4,CMP0005,STORE_018,2023-01,Influencer,8000,29342800


In [4668]:
marketing_campaigns.info()
#اكتشفت مشكله ان عمود الBudget_SAR فيه اختلاف في صياغة القيم كذلك نوعها سترنق وليس انتجر

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 218 entries, 0 to 217
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   Campaign_ID    218 non-null    object
 1   Store_ID       218 non-null    object
 2   Month          218 non-null    object
 3   Campaign_Type  218 non-null    object
 4   Budget_SAR     218 non-null    object
 5   Impressions    218 non-null    int64 
dtypes: int64(1), object(5)
memory usage: 10.3+ KB


In [4669]:
marketing_campaigns.isnull().sum()

Campaign_ID      0
Store_ID         0
Month            0
Campaign_Type    0
Budget_SAR       0
Impressions      0
dtype: int64

In [4670]:

marketing_campaigns.describe()

,Impressions
count,218
mean,"941,252"
std,"4,982,861"
min,"10,082"
25%,"130,957"
50%,"257,210"
75%,"372,085"
max,"47,762,300"


In [4671]:
#Campaign_Type حساب القيم الفريدة في  و Campaign_ID
unique_Campaign_Type = marketing_campaigns['Campaign_Type'].unique()
unique_Campaign_ID = marketing_campaigns['Campaign_ID'].nunique()
unique_Store_ID = marketing_campaigns['Store_ID'].nunique()

# عرض النتائج
print(f"  unique_Campaign_ID : {unique_Campaign_ID}")
print(f"  unique_Campaign_Type : {unique_Campaign_Type}")
print(f"  unique_Store_ID : {unique_Store_ID}")
##هنا نكتشف ان عندنا 46 متجر وفي الحقيقه يوجد فقط 45 متجر نتاكد من المتجر الزايد
# عرض القيم الفريدة في Store_ID



  unique_Campaign_ID : 218
  unique_Campaign_Type : ['Influencer' 'Outdoor' 'Radio' 'Email' 'TV' 'Social Media']
  unique_Store_ID : 46


In [4672]:
unique_store_ids = marketing_campaigns['Store_ID'].unique()
print(f"Unique Store_ID values: {unique_store_ids}")
#اكتشفت عندي store_999 وهو متجر ليس موجود او انه وهمي 
#نشوف كم مره تكرر
# عرض الصفوف التي تحتوي على Store_ID "STORE_999"
store_999_rows = marketing_campaigns[marketing_campaigns['Store_ID'] == 'STORE_999']

# عرض الصفوف
print(f"Rows with Store_ID 'STORE_999':\n{store_999_rows}")

Unique Store_ID values: ['STORE_033' 'STORE_002' 'STORE_045' 'STORE_018' 'STORE_030' 'STORE_014'
 'STORE_024' 'STORE_005' 'STORE_041' 'STORE_031' 'STORE_019' 'STORE_040'
 'STORE_029' 'STORE_027' 'STORE_003' 'STORE_007' 'STORE_015' 'STORE_013'
 'STORE_042' 'STORE_999' 'STORE_009' 'STORE_037' 'STORE_023' 'STORE_001'
 'STORE_011' 'STORE_010' 'STORE_020' 'STORE_043' 'STORE_008' 'STORE_006'
 'STORE_044' 'STORE_021' 'STORE_016' 'STORE_035' 'STORE_025' 'STORE_032'
 'STORE_012' 'STORE_036' 'STORE_004' 'STORE_034' 'STORE_038' 'STORE_022'
 'STORE_039' 'STORE_026' 'STORE_028' 'STORE_017']
Rows with Store_ID 'STORE_999':
    Campaign_ID   Store_ID    Month Campaign_Type Budget_SAR  Impressions
23      CMP0024  STORE_999  2023-02    Influencer      14000       486877
55      CMP0056  STORE_999  2023-05    Influencer        43K       296235
62      CMP0063  STORE_999  2023-06  Social Media  27000 SAR       406964
111     CMP0112  STORE_999  2023-10         Email      38000       240248
159     CMP01

In [4673]:
# التحقق من وجود مسافات فارغة فقط في الأعمدة النصية
whitespace_rows = marketing_campaigns[marketing_campaigns.select_dtypes(include=['object']).apply(lambda x: x.str.contains('^\s*$', na=False)).any(axis=1)]

# عرض الصفوف التي تحتوي على مسافات فارغة في الأعمدة النصية
print("\nRows with White Spaces in Any Text Column:")
print(whitespace_rows)


Rows with White Spaces in Any Text Column:
Empty DataFrame
Columns: [Campaign_ID, Store_ID, Month, Campaign_Type, Budget_SAR, Impressions]
Index: []


In [4674]:
# تحويل عمود Month إلى نوع تاريخ (datetime)
marketing_campaigns['Month'] = pd.to_datetime(marketing_campaigns['Month'], format='%Y-%m')

# التحقق من نوع البيانات بعد التحويل
print(f"Data type of Month after conversion: {marketing_campaigns['Month'].dtype}")

Data type of Month after conversion: datetime64[ns]


In [4675]:
# حذف الصفوف التي تحتوي على Store_ID "STORE_999"
marketing_campaigns = marketing_campaigns[marketing_campaigns['Store_ID'] != 'STORE_999']

# عرض عدد الصفوف بعد الحذف
print(f" rows after delete  : {marketing_campaigns.shape[0]}")

 rows after delete  : 208


In [4676]:
# عرض أقدم وأحدث تاريخ
earliest_date = marketing_campaigns['Month'].min()
latest_date = marketing_campaigns['Month'].max()

print(f"\nEarliest Date in the data: {earliest_date}")
print(f"Latest Date in the data: {latest_date}")


Earliest Date in the data: 2023-01-01 00:00:00
Latest Date in the data: 2024-06-01 00:00:00


In [4677]:
# تنظيف عمود Budget_SAR
import re

def clean_budget(value):
    value = str(value).strip()
    value = value.upper().replace('SAR', '').replace(',', '').strip()
    if value.endswith('K'):
        return int(float(value[:-1]) * 1000)
    return int(float(value))

marketing_campaigns['Budget_SAR'] = marketing_campaigns['Budget_SAR'].apply(clean_budget)

print(marketing_campaigns[['Campaign_ID', 'Budget_SAR']].to_string())
print("\nنوع البيانات:", marketing_campaigns['Budget_SAR'].dtype)

    Campaign_ID  Budget_SAR
0       CMP0001       46000
1       CMP0002       26000
2       CMP0003       40000
3       CMP0004       47000
4       CMP0005        8000
5       CMP0006       10000
6       CMP0007       15000
7       CMP0008       30000
8       CMP0009       11000
9       CMP0010       18000
10      CMP0011       25000
11      CMP0012       18000
12      CMP0013       42000
13      CMP0014       47000
14      CMP0015       24000
15      CMP0016       22000
16      CMP0017       17000
17      CMP0018       12000
18      CMP0019       50000
19      CMP0020        5000
20      CMP0021       23000
21      CMP0022       11000
22      CMP0023       42000
24      CMP0025       29000
25      CMP0026       33000
26      CMP0027       31000
27      CMP0028       17000
28      CMP0029       25000
29      CMP0030       50000
30      CMP0031       24000
31      CMP0032       15000
32      CMP0033       20000
33      CMP0034       25000
34      CMP0035       11000
35      CMP0036     

In [4678]:
marketing_campaigns.to_csv('Cleaned_marketing_campaigns.csv', index=False)


In [4679]:
#الملف الخامس

In [4680]:
store_complaints.shape

(500, 6)

In [4681]:
store_complaints.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Complaint_ID    500 non-null    object 
 1   Store_ID        50 non-null     object 
 2   Date            50 non-null     object 
 3   Complaint_Type  44 non-null     object 
 4   Resolved        31 non-null     object 
 5   Notes           0 non-null      float64
dtypes: float64(1), object(5)
memory usage: 23.6+ KB


In [4682]:
store_complaints.head()

,Complaint_ID,Store_ID,Date,Complaint_Type,Resolved,Notes
0,COMP0001,NaN,NaN,NaN,NaN,NaN
1,COMP0002,NaN,NaN,NaN,NaN,NaN
2,COMP0003,NaN,NaN,NaN,NaN,NaN
3,COMP0004,NaN,NaN,NaN,NaN,NaN
4,COMP0005,NaN,NaN,NaN,NaN,NaN


In [4683]:
store_complaints.isnull().sum()

Complaint_ID        0
Store_ID          450
Date              450
Complaint_Type    456
Resolved          469
Notes             500
dtype: int64

In [4684]:
# حساب القيم الفريدة    
unique_Complaint_Type = store_complaints['Complaint_Type'].unique()
unique_Resolved = store_complaints['Resolved'].unique()
unique_Complaint_ID = store_complaints['Complaint_ID'].nunique()
unique_Store_ID = store_complaints['Store_ID'].nunique()

# عرض النتائج
print(f"  unique_Complaint_Type : {unique_Complaint_Type}")
print(f"  unique_Resolved : {unique_Resolved}")
print(f"  unique_Complaint_ID : {unique_Complaint_ID}")
print(f"  unique_Store_ID : {unique_Store_ID}")

#الان نحذف جميع الصفوف ال450 التي ليس لها  store ID ويتبقى 50 صف نتعامل معه 

  unique_Complaint_Type : [nan 'Other' 'Cleanliness' 'Product' 'Service']
  unique_Resolved : [nan 'Yes' 'No' 'Pending']
  unique_Complaint_ID : 500
  unique_Store_ID : 30


In [4685]:
# حذف الصفوف التي ليس لديها قيمة في عمود Store_ID
store_complaints = store_complaints.dropna(subset=['Store_ID'])

# عرض عدد الصفوف بعد الحذف
print(f"Number of rows after removing rows with missing Store_ID: {store_complaints.shape[0]}")
# حذف عمود Notes بالكامل
store_complaints = store_complaints.drop(columns=['Notes'])
# عرض بعض البيانات للتأكد
store_complaints.head(99)

Number of rows after removing rows with missing Store_ID: 50


,Complaint_ID,Store_ID,Date,Complaint_Type,Resolved
8,COMP0009,STORE_045,2023-07-13,Other,Yes
12,COMP0013,STORE_009,2023-04-27,Cleanliness,NaN
13,COMP0014,STORE_015,2023-12-10,Product,No
41,COMP0042,STORE_020,2024-06-22,Product,No
85,COMP0086,STORE_034,2024-01-11,Service,Yes
92,COMP0093,STORE_029,2023-01-20,Service,Yes
100,COMP0101,STORE_030,2024-06-26,Product,NaN
108,COMP0109,STORE_016,2023-02-08,Other,Yes
121,COMP0122,STORE_011,2023-02-01,Other,Yes
126,COMP0127,STORE_008,2023-05-15,Other,NaN


In [4686]:
store_complaints.shape

(50, 5)

In [4687]:
store_complaints.isnull().sum()

Complaint_ID       0
Store_ID           0
Date               0
Complaint_Type     6
Resolved          19
dtype: int64

In [4688]:
# تعبئة القيم الناقصة
store_complaints['Complaint_Type'] = store_complaints['Complaint_Type'].fillna('Other')
store_complaints['Resolved'] = store_complaints['Resolved'].fillna('Unknown')

# التحقق
print(store_complaints.isnull().sum())
print()
print(store_complaints[['Complaint_Type', 'Resolved']].value_counts())

Complaint_ID      0
Store_ID          0
Date              0
Complaint_Type    0
Resolved          0
dtype: int64

Complaint_Type  Resolved
Cleanliness     Unknown     6
Product         No          6
Other           Unknown     5
                Yes         5
Product         Unknown     4
Service         No          4
                Unknown     4
Other           No          3
Product         Pending     3
Service         Yes         3
Cleanliness     Pending     2
Product         Yes         2
Cleanliness     Yes         1
Other           Pending     1
Service         Pending     1
dtype: int64


In [4689]:
store_complaints.head(50)

,Complaint_ID,Store_ID,Date,Complaint_Type,Resolved
8,COMP0009,STORE_045,2023-07-13,Other,Yes
12,COMP0013,STORE_009,2023-04-27,Cleanliness,Unknown
13,COMP0014,STORE_015,2023-12-10,Product,No
41,COMP0042,STORE_020,2024-06-22,Product,No
85,COMP0086,STORE_034,2024-01-11,Service,Yes
92,COMP0093,STORE_029,2023-01-20,Service,Yes
100,COMP0101,STORE_030,2024-06-26,Product,Unknown
108,COMP0109,STORE_016,2023-02-08,Other,Yes
121,COMP0122,STORE_011,2023-02-01,Other,Yes
126,COMP0127,STORE_008,2023-05-15,Other,Unknown


In [4690]:

# تحويل العمود Date إلى نوع datetime مع توحيد الصيغ في DataFrame الصحيح
store_complaints['Date'] = pd.to_datetime(store_complaints['Date'], errors='coerce')

In [4691]:
# عرض أقدم وأحدث تاريخ
earliest_date = store_complaints['Date'].min()
latest_date = store_complaints['Date'].max()

print(f"\nEarliest Date in the data: {earliest_date}")
print(f"Latest Date in the data: {latest_date}")


Earliest Date in the data: 2023-01-03 00:00:00
Latest Date in the data: 2024-06-26 00:00:00


In [4692]:
store_complaints.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 50 entries, 8 to 493
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Complaint_ID    50 non-null     object        
 1   Store_ID        50 non-null     object        
 2   Date            50 non-null     datetime64[ns]
 3   Complaint_Type  50 non-null     object        
 4   Resolved        50 non-null     object        
dtypes: datetime64[ns](1), object(4)
memory usage: 2.3+ KB


In [4693]:
# التحقق من وجود مسافات فارغة فقط في الأعمدة النصية
whitespace_rows = store_complaints[store_complaints.select_dtypes(include=['object']).apply(lambda x: x.str.contains('^\s*$', na=False)).any(axis=1)]

# عرض الصفوف التي تحتوي على مسافات فارغة في الأعمدة النصية
print("\nRows with White Spaces in Any Text Column:")
print(whitespace_rows)


Rows with White Spaces in Any Text Column:
Empty DataFrame
Columns: [Complaint_ID, Store_ID, Date, Complaint_Type, Resolved]
Index: []


In [4694]:
store_complaints.to_csv('Cleaned_store_complaints.csv', index=False)


In [4695]:
Cleaned_store_complaints = pd.read_csv('Cleaned_store_complaints.csv')


In [4696]:
Cleaned_store_complaints.shape

(50, 5)